# Exercise: Development

Welcome to the second notebook of the advanced topic module. 

In this notebook, we will practice the development cycle for creating machine learning models, and we will use the machine learning tracker, Weights and Biases, to log and automate our experiments as seen in the video. We will continue with the air quality dataset to exemplify some of the explained practices. 

Weights & Biases is a machine learning platform for developers to build better models faster. It allows to track experiments, version and iterate on datasets, evaluate model performance, reproduce models, visualize results and spot regressions, and share findings with colleagues. 

In this tutorial, we will demonstrate how you can easily execute advanced hyperparameter sweeps in three simple steps using Weights and Biases.

<center><img src="https://wandb.me/logo-im-png" width="400" alt="Weights & Biases" /></center>


For this notebook, we will create and train our machine learning models using PyTorch. Also, we will define a threshold of 0.80 accuracy as success criterium.

## Libraries

For running this notebook, we will need the following libraries

In [ ]:
import pickle
from sklearn.metrics import r2_score
import random
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, random_split

## Importing Weights and Biases

First of all, we are going to set up Weights and Biases. We will need to log in to our account. If you don't have an account, you can create one [here](https://wandb.ai/site).


If you've never used Weights & Biases before, the call to `login` will give you a link to sign up for an account.

W&B is free to use for personal and academic projects.

In [ ]:
import wandb

In [ ]:
wandb.login()

You can test that your installation is working by running the following cell

In [ ]:
# start a new wandb run to track this script
wandb.init(
    # set the wandb project where this run will be logged
    project="my-test-project",
    
    # track hyperparameters and run metadata
    config={
    "learning_rate": 0.02,
    "architecture": "ANN",
    "dataset": "Air_quality",
    "epochs": 100,
    }
)

# simulate training. We use random numbers to simulate a training loop here. 
epochs = 100
for epoch in range(2, epochs):
    acc = epoch * random.random()
    loss = 60 - epoch * random.random()
    
    # log metrics to wandb
    wandb.log({"acc": acc, "loss": loss})
    
# Finish the wandb run, necessary in notebooks
wandb.finish()

You should be able to see the history of the accuracy and loss. You can also see the model architecture and a link for viewing the run data. You can follow that link to see the run in the W&B dashboard.

Now that we have succesfully set up weights and biases, we can start with the development cycle.

## Loading the datafiles and scalers

These are the datasets and scalers that we saved in previous notebook.

**Exercise 1.**

Using the example for the training data, load the validation data. Use the variable names `validation features` and `validation targets`.

In [ ]:
# Loading training data
training_features = pd.read_csv('Scaled_Training_features.csv')
training_targets = pd.read_csv('Scaled_Training_targets.csv')
 
# Loading validation data
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #


**Exercise 2:**

Following the example of the X scaler, load the y_scaler.

In [ ]:
# Load X_scaler
with open('X_scaler.pk', 'rb') as file:
    X_scaler = pickle.load(file)

# Load y_scaler
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #


PyTorch uses Tensor as datatype. Therefore, we convert all the data to tensors.

In [ ]:
# Training data
training_features = torch.tensor(training_features.values, dtype=torch.float32)
training_targets = torch.tensor(training_targets.values, dtype=torch.float32)
 
# Validation data
validation_features = torch.tensor(validation_features.values, dtype=torch.float32)
validation_targets = torch.tensor(validation_targets.values, dtype=torch.float32)

In [ ]:
class RegressionModel(nn.Module):
    def __init__(self, input_size):
        super(RegressionModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)
 
    def forward(self, x):
        return self.linear(x)

In [ ]:
model = RegressionModel(input_size=training_features.shape[1])

### Defining the optimizer and loss function

**Optimizer** defines how the weights of the neural network are to be updated. It takes in model parameters and learning rate as the input arguments. Here we will use Stochastic Gradient Descent (SGD), given by the equation,

$$w^{new}:=w^{old}-\eta \nabla L(w^{new})$$

where, $\eta$ is the learning rate, $w^{new}$ and $w^{old}$ are the new and the old weights of the neural connections and $L$ is the objective function.


**Loss function** is the objective that you want to minimize. Since, we will develop a regression, a common loss function to use is the Mean Squarred Error (MSE), given by the equation

$$MSE=\frac{1}{n}\sum_{i=1}^{n}(\hat{y}-y)^2$$

where $\hat{y}$ is the predicted value and $y$ is the actual value of the target variable.

**Additional metric:**

In addition, we will use the coeficient of determination, usually named $R^2$ as a metric to indicate how good is the fitting of our model respect to the average value.

$$R^2 = 1 - \frac{(\hat{y} - y)^2}{(\bar{y} - y)^2}$$
where $\hat{y}$ is the predicted value, $\bar{y}$ is the average value of $y$, which is the actual value of the target variable.

In [ ]:
# Define loss function and optimizer
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1)
# For the R2 metric we will use the predefined function from Scikit learn `r2_score`.

## Training the multilinear regression

Now, we can train the multilinear regression with the following code

**Exercise 3:**

1. As seen in the example above, initialize the W&B tracking (using `wandb.init`). Set the following properties: 
- project
- config
  - architecture
  - dataset
  - epochs

For the epochs, you can set a value of 20 to quickly run your code. Nevertheless, you can use a different value. For the rest of the properties, you can use the strings that you find more suitable.

2. Inside the for loop. Use `wandb.log` to record the loss, the r2 score in training, and the  r2 score in validation.

In [ ]:
# start a new wandb run to track this script
wandb.init(
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #

)

epochs = wandb.config['epochs']

for epoch in range(epochs):
    
    outputs = model(training_features)
    loss = loss_function(outputs, training_targets)
 
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    
    r2_training = r2_score(training_targets.detach().numpy(), outputs.detach().numpy())
    
    outputs_validation = model(validation_features)
    r2_validation = r2_score(validation_targets.detach().numpy(), outputs_validation.detach().numpy())
    
    
    # log metrics to wandb
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    
# Finish the wandb run, necessary in notebooks
wandb.finish()

We see that our validation score is around 0.75, which is not bad. However, it is still inferior than our sucess criteria (0.80). 

We think that a model with more parameters could have higher accuracy. Therefore, we will try a different model.

## Artificial Neural Network

For this new iteration of the model, we will use an artificial neural network. In this case, the ANN has more hyperparameters than the previous model. For exploring the multiple possibilities, we will use a random search with a W&B sweep.

### Hyperparameter Sweeps using W&B

Exploring high-dimensional hyperparameter spaces in search of the most effective model can quickly become challenging. Hyperparameter sweeps offer a systematic and effective approach to train, evaluate models and choose the best performing one. This is achieved by automatically exploring various combinations of hyperparameter values (hidden dimensions, hidden layers, learning rate, etc.) to determine the optimal configuration.

### Sweeps: An Overview

For running a hyperparameter sweep with Weights & Biases you need to follow these 3 steps:

1. **Define the sweep:** we do this by creating a dictionary (or a [YAML file](https://docs.wandb.com/library/sweeps/configuration)) that specifies the parameters to search through, the search strategy, and the optimization metric.

2. **Initialize the sweep:** We initialize the sweep and pass in the dictionary of sweep configurations:
`sweep_id = wandb.sweep(sweep_config)`

1. **Run the sweep agent:** We call `wandb.agent()` and pass the `sweep_id` to run, along with a function that defines your model architecture and trains it:
`wandb.agent(sweep_id, function=train)`

Below, we'll walk through these 3 steps in more detail.

### Resources

You can explore other resources:
- Weights and biases official [video tutorial](http://wandb.me/sweeps-video)!
- [Sweeps docs →](https://docs.wandb.ai/sweeps)
- [Launching from the command line →](https://www.wandb.com/articles/hyperparameter-tuning-as-easy-as-1-2-3)


## Step 1️. Define the Sweep

Fundamentally, a Sweep combines a strategy for trying many hyperparameter values with the code that evalutes them.

You need to _define your strategy_
in the form of a [configuration](https://docs.wandb.com/sweeps/configuration).

When you're setting up a Sweep in a notebook like this, that config object is a nested dictionary.
When you run a Sweep via the command line, the config object is a [YAML file](https://docs.wandb.com/sweeps/quickstart#2-sweep-config).

### Choose a `method`

The first thing we need to define is the `method`
for choosing new parameter values.

W&B provides the following search `methods`:
*   **`grid` Search** – Iterate over every combination of hyperparameter values. Very effective, but can be computationally costly.
*   **`random` Search** – Select each new combination at random according to provided `distribution`s.
*   **`bayes`ian Search** – Create a probabilistic model of metric score as a function of the hyperparameters, and choose parameters with high probability of improving the metric.

We'll use a random search for now. Let's declare that in our config variable.

In [ ]:
sweep_config = {
    'method': 'random'
    }

Let's record as well the metric that we want to minimize.

In [ ]:
metric = {
    'name': 'loss',
    'goal': 'minimize'
    }
sweep_config['metric'] = metric

### Define the hyperparameter ranges

After deciding on a mechanism to experiment with different hyperparameter values, you need to define the specific hyperparameters you want to tune. This process is usually straightforward, as it involves naming the hyperparameter and listing the acceptable values.

For example, when selecting an optimization algorithm for your neural network, there's a finite set of options. Here, we focus on the two most widely used choices: adam and sgd. Even for hyperparameters with potentially limitless possibilities, it typically makes sense to investigate only a few carefully chosen values, as we do with the hidden layer size and dropout rate.

**Exercise 4:**

Define the values for the hyperparameter 'number_hidden_layers'. You can try between 1 and 5 layers. 

In [ ]:
parameters_dict = {
    'hidden_dimensions': {
        'values': [8, 16, 32, 64]
        },
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #
    }

sweep_config['parameters'] = parameters_dict

It's quite usual to come across hyperparameters that we don't want to change during this hyperparameter sweep but still need to define within our sweep_config. In these instances, we directly set the preferred value.

In [ ]:
parameters_dict.update({
    'epochs': {
        'value': 80}
    })

We are ready to run the random search. Each hyperparameter will be drawn with equal probability.

In case we wanted to run a random search with different distribution, we would need to define the distributions for each of the hyperparameters.

You can specify a named `distribution`, plus its parameters, for example the mean $\mu$ and standard deviation $\sigma$ of a `normal` distribution.

See more on how to set the distributions of your random variables [here](https://docs.wandb.com/sweeps/configuration#distributions).

Now, sweep_config has become a nested dictionary that clearly defines the hyperparameters we want to investigate and the approach we'll use to evaluate them.

In [ ]:
print(sweep_config)

There are more configuration options that you can explore [here](https://docs.wandb.com/sweeps/configuration).

## Step 2. Initialize the Sweep

Once you've defined the search strategy, it's time to set up something to implement it.

The administrator of the Sweep is known as the _Sweep Controller_.
As each run completes, it will issue a new set of instructions describing a new run to execute.
These instructions are picked up by _agents_ who actually perform the runs.

In a typical Sweep, the Controller lives on W&B machine, while the agents who complete runs live on _your_ machine(s) or cluster.

This division of labor makes it very convenient to scale up Sweeps by adding more machines to run agents!

We can wind up a Sweep Controller by calling `wandb.sweep` with the appropriate `sweep_config` and `project` name.

This function returns a `sweep_id` that we will later user to assign agents to this Controller.

**Exercise 5**

Define the `sweep_id` variable. For this, use the function `wandb.sweep`. As parameters give the sweep_config and `project =  <project name you chose>`

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

## Step 3️. Run the Sweep agent

First, we need to define a function that will train our model. This function will be called by the sweep agent.

This is a standard procedure of training a neural network. First, we define the ANN model, then we define a function that creates different ANNs based on the chosen hyperparameters. 
Then, we define the loss function. 
Finally, we train the model while logging the results.

In [ ]:
class ANN(nn.Module):
    def __init__(self, input_size, number_hidden_layers, hidden_dimensions):
        super(ANN, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_dimensions))  # First hidden layer
        for _ in range(number_hidden_layers - 1):
            self.layers.append(nn.Linear(hidden_dimensions, hidden_dimensions))  # Additional hidden layers
        self.layers.append(nn.Linear(hidden_dimensions, 1))  # Output layer

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.tanh(layer(x))  # Apply tanh activation to hidden layers
        x = self.layers[-1](x)  # Output layer
        return x

In addition to the class that defines the ANN, we need a function that builds different ANNs based on the number of hidden layers and number of hidden dimensions. In this case, we will assume that all of the hidden layers will have the same amount of hidden dimensions. 

In [ ]:
def build_network(number_hidden_layers, hidden_dimensions):
    network = ANN(input_size=training_features.shape[1], 
                  number_hidden_layers=number_hidden_layers, 
                  hidden_dimensions=hidden_dimensions)

    return network

We need to create a function `train` that receives as parameter a config variable. This config variable is where W&B chooses with which hyperparameters the model should be trained.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_function = nn.MSELoss()

def train(config=None):
    # Initialize a new wandb run
    with wandb.init(config=config):
        # If called by wandb.agent, as below,
        # this config will be set by Sweep Controller
        config = wandb.config

        network = build_network(config.number_hidden_layers, config.hidden_dimensions).to(device)
        optimizer = optim.SGD(network.parameters(), lr=0.1, momentum=0.9)

        for epoch in range(config.epochs):
            outputs = network(training_features)
            loss = loss_function(outputs, training_targets)
        
            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            r2_training = r2_score(training_targets.detach().numpy(), outputs.detach().numpy())
            
            outputs_validation = network(validation_features)
            r2_validation = r2_score(validation_targets.detach().numpy(), outputs_validation.detach().numpy())
            
            # log metrics to wandb
            wandb.log({"r2_score_training": r2_training, 
                    "r2_score_validation": r2_validation, 
                    "loss": loss})

**Exercise 6:**

Run the hyperparameter sweep. For this, use the command `wandb.agent`, which receives the following parameters:
- Variable `sweep_id`, 
- Function `train`: Function defined above that trains a machine learning model according to the hyperparameters in config.
- Value `count` number of models that we want to train.  

The following cell is the responsible for indicating to the controller that this machine will be running the sweep.

**Note:** This cell can take a while to run depending on how many iterations you specified in the sweep count.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

Notice that if you were to run this notebook in another computer, the W&B server would make sure that different combinations of hyperparameters are assigned to different computers. So, you can do this hyperparameter optimization in parallel!

## Train the best model

**Exercise 7:**

Open your W&B dashboard and see the sweep running. 
Once it finishes, choose the best model (model with the lowest validation loss) and train it again in the cells below to get the final results. 

The best model is the one with the following hyperparameters:
- number_of_hidden_layers: __
- hidden_dimensions: __

Add them to the `selected_config` variable.

In [ ]:
selected_config = {

    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
}

We create a model with these hyperparameters and train it again.

In [ ]:
model = build_network(selected_config['number_of_hidden_layers'], selected_config['hidden_dimensions']).to(device)

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
wandb.init(
    # set the wandb project where this run will be logged
    project="air-quality-prediction",
    
    # track hyperparameters and run metadata
    config={
    "architecture": "Artificial Neural Network",
    "dataset": "Air_quality",
    "epochs": 80,
    }
)
epochs = wandb.config['epochs']

for epoch in range(epochs):
    outputs = model(training_features)
    loss = loss_function(outputs, training_targets)
 
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    r2_training = r2_score(training_targets.detach().numpy(), outputs.detach().numpy())
    outputs_validation = model(validation_features)
    r2_validation = r2_score(validation_targets.detach().numpy(), outputs_validation.detach().numpy())
    
    
    # log metrics to wandb
    wandb.log({"r2_score_training": r2_training, 
               "r2_score_validation": r2_validation, 
               "loss": loss})
    
# Finish the wandb run, necessary in notebooks
wandb.finish()

Finally, we can use the test set to evaluate the model. 

Notice that only up to this point te check the test set. We don't use this set when training or hyperoptimizing the model.

In [ ]:
# Loading testing data
testing_features = pd.read_csv('Scaled_Testing_Features.csv')
testing_targets = pd.read_csv('Scaled_Testing_targets.csv')

# Testing data
testing_features = torch.tensor(testing_features.values, dtype=torch.float32)
testing_targets = torch.tensor(testing_targets.values, dtype=torch.float32)

In [ ]:
testing_outputs = model(testing_features)
loss = loss_function(testing_outputs, testing_targets)

r2 = r2_score(testing_targets.detach().numpy(), testing_outputs.detach().numpy())
print(r2)

This number represents the accuracy of the model in the test set. This is, data not seen by the model during training or validation. Therefore, this is the best estimate of the accuracy of the model.

## Export the model

Once we are satisfied with the results, we can export the model to use it in the future.

In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')

We could have continued with the development cycle and try different models. However, we will stop here for the sake of time. In the next notebook, we will see how to deploy the model with a small usage application.